In [ ]:
!rm -rf /kaggle/working/unsloth_compiled_cache

In [ ]:
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade trl transformers datasets accelerate peft

In [ ]:
import os

print("Folders in /kaggle/input/:")
for item in os.listdir('/kaggle/input/'):
    print(f" - {item}")
    
    # If it's a directory, print what is inside it
    folder_path = os.path.join('/kaggle/input/', item)
    if os.path.isdir(folder_path):
        for sub_item in os.listdir(folder_path):
            print(f"    -> /kaggle/input/{item}/{sub_item}")

In [ ]:
import os

print("Searching for JSONL files...")
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.jsonl'):
            print(os.path.join(root, file))

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
import os

# 1. Load the 1.5B Model
max_seq_length = 2048 
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True, # Crucial for fitting in Kaggle memory
)

# Apply LoRA for efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 2. Format the Prompts
prompt_template = """Below is an instruction that describes a programming task. Write a response that appropriately completes the request.
### Instruction:
{prompt}
### Response:
{response}"""

def format_prompts(examples):
    prompts = examples["prompt"]
    # If long_cot exists, use it. Otherwise, use short_cot.
    responses = [l if l else s for l, s in zip(examples.get("long_cot", [""]*len(prompts)), examples["short_cot"])]
    
    texts = []
    for p, r in zip(prompts, responses):
        texts.append(prompt_template.format(prompt=p, response=r))
    return { "text" : texts }

# 3. Execute the Curriculum
epochs_data = [
    "/kaggle/input/datasets/haris77777/thesis-dynamic-mix-curriculum/epoch_1.jsonl",
    "/kaggle/input/datasets/haris77777/thesis-dynamic-mix-curriculum/epoch_2.jsonl",
    "/kaggle/input/datasets/haris77777/thesis-dynamic-mix-curriculum/epoch_3.jsonl"
]

for epoch_idx, dataset_path in enumerate(epochs_data):
    print(f"\n{'='*40}")
    print(f"STARTING CURRICULUM EPOCH {epoch_idx + 1}")
    print(f"{'='*40}\n")
    
    dataset = load_dataset("json", data_files=dataset_path, split="train")
    dataset = dataset.map(format_prompts, batched = True)
    
    trainer = SFTTrainer(
        model = model,
        processing_class = tokenizer,
        train_dataset = dataset,
        args = SFTConfig(
            dataset_text_field = "text",
            max_seq_length = max_seq_length,
            dataset_num_proc = 2,
            padding_free = False,          # <--- THE FIX
            per_device_train_batch_size = 2,
            gradient_accumulation_steps = 4,
            warmup_steps = 5,
            num_train_epochs = 1, 
            learning_rate = 2e-4,
            fp16 = not torch.cuda.is_bf16_supported(),
            bf16 = torch.cuda.is_bf16_supported(),
            logging_steps = 10,
            optim = "adamw_8bit",
            weight_decay = 0.01,
            output_dir = f"outputs_epoch_{epoch_idx+1}",
            report_to="none" 
        ),
    )
    
    trainer_stats = trainer.train()
    
    # Save the adapter weights after each epoch so you don't lose progress!
    save_path = f"lora_model_epoch_{epoch_idx+1}"
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Checkpoint saved to {save_path}")

print("Training Complete! All epochs finished.")

In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
import torch
import json
import re
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

# 1. Load Your Final Thesis Model
model_path = "./lora_model_epoch_3" 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model) # Speeds up generation 2x

# 2. Load the HumanEval Benchmark
humaneval = load_dataset("openai_humaneval", split="test")

# 3. Extraction Function (Crucial for getting a Pass@1 score)
def extract_python_code(text):
    pattern = r"```python\n(.*?)\n```"
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        return matches[-1] 
    return text 

# 4. Generate the Completions
output_file = "curriculum_humaneval_results.jsonl" 
prompt_template = """Below is an instruction that describes a programming task. Write a response that appropriately completes the request.
### Instruction:
{prompt}
### Response:
"""

print(f"Generating responses for {len(humaneval)} problems...")

with open(output_file, "w") as f:
    for problem in tqdm(humaneval):
        task_id = problem["task_id"]
        prompt = problem["prompt"]
        
        formatted_prompt = prompt_template.format(prompt=prompt)
        inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
        
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1024, 
            temperature=0.2, 
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
        
        full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        generated_text = full_response.split("### Response:\n")[-1]
        clean_code = extract_python_code(generated_text)
        
        result_dict = {
            "task_id": task_id,
            "completion": clean_code
        }
        f.write(json.dumps(result_dict) + "\n")

print("Generation complete! Ready for Pass@1 execution.")

In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
import torch
import json
import re
from tqdm import tqdm

# 1. Change the model path back to the Hugging Face base model
model_path = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model) # Speeds up generation 2x

# 2. Load the HumanEval Benchmark
humaneval = load_dataset("openai_humaneval", split="test")

# 3. Extraction Function (Crucial for getting a Pass@1 score)
def extract_python_code(text):
    pattern = r"```python\n(.*?)\n```"
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        return matches[-1] 
    return text 

# 4. Generate the Completions
output_file = "base_humaneval_results.jsonl"
prompt_template = """Below is an instruction that describes a programming task. Write a response that appropriately completes the request.
### Instruction:
{prompt}
### Response:
"""

print(f"Generating responses for {len(humaneval)} problems...")

with open(output_file, "w") as f:
    for problem in tqdm(humaneval):
        task_id = problem["task_id"]
        prompt = problem["prompt"]
        
        formatted_prompt = prompt_template.format(prompt=prompt)
        inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
        
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1024, 
            temperature=0.2, 
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
        
        full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        generated_text = full_response.split("### Response:\n")[-1]
        clean_code = extract_python_code(generated_text)
        
        result_dict = {
            "task_id": task_id,
            "completion": clean_code
        }
        f.write(json.dumps(result_dict) + "\n")

print("Generation complete! Ready for Pass@1 execution.")

In [ ]:
# Run the evaluation on your Base Model
!evaluate_functional_correctness base_humaneval_results.jsonl

# Run the evaluation on your Curriculum LoRA Model
!evaluate_functional_correctness curriculum_humaneval_results.jsonl